# SAP AI: Kaggle training and human Arena

This notebook is the Kaggle-native counterpart to `sapai_colab_training.ipynb`. Before running it, enable **Internet** in Notebook Settings, select a **GPU accelerator** for training, and add a `DATABASE_URL` Kaggle secret if you are not attaching an existing `boards.jsonl` dataset.

All writable artifacts go under `/kaggle/working`. Keep the human benchmark disabled during **Save & Run All**; launch it only in an interactive editor session.

In [ ]:
REPO_URL = "https://github.com/lgtyqz/sapai-python.git"
BRANCH = "main"
KAGGLE_RUN_DIR = "/kaggle/working/sapai-runs/run-001"
KAGGLE_PRIOR_RUN_DIR = ""  # Optional attached output directory under /kaggle/input.
BOARDS_JSONL = ""  # Optional attached boards.jsonl; blank auto-detects one.
BOARD_EXPORT_LIMIT = 10000
PACK = "Turtle"
SEED = 2026
REQUIRE_GPU = True
RUN_FULL_TRAINING = False
RUN_HUMAN_BENCHMARK = False
HUMAN_BENCHMARK_DIR = "/kaggle/working/sapai-human/benchmark-001"
KAGGLE_PRIOR_HUMAN_DIR = ""  # Optional attached human directory under /kaggle/input.
HUMAN_PARTICIPANT_ALIAS = "anonymous"
HUMAN_SEED = 2026

assert REPO_URL, "REPO_URL cannot be empty."
assert BRANCH, "BRANCH cannot be empty."
assert KAGGLE_RUN_DIR.startswith('/kaggle/working/'), (
    'KAGGLE_RUN_DIR must be inside /kaggle/working so Kaggle can retain it as output.'
)
assert HUMAN_BENCHMARK_DIR.startswith('/kaggle/working/'), (
    'HUMAN_BENCHMARK_DIR must be inside /kaggle/working.'
)
assert HUMAN_PARTICIPANT_ALIAS.strip(), "HUMAN_PARTICIPANT_ALIAS cannot be empty."
assert 2 <= BOARD_EXPORT_LIMIT <= 10000

## Repository and prior outputs

Kaggle inputs are read-only. To continue a prior run, attach the earlier notebook output through **Add Input**, set `KAGGLE_PRIOR_RUN_DIR` to the attached run directory, and rerun this cell. It copies the checkpoint tree into writable storage only when the destination is still empty. The same mechanism is available for a human benchmark directory.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

working_root = Path('/kaggle/working')
input_root = Path('/kaggle/input')
repo = working_root / 'sapai-python'
if repo.exists() and not (repo / '.git').is_dir():
    raise RuntimeError(f'{repo} exists but is not a Git checkout.')
if not repo.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo)],
        check=True,
    )
else:
    origin = subprocess.run(
        ['git', '-C', str(repo), 'remote', 'get-url', 'origin'],
        check=True, capture_output=True, text=True,
    ).stdout.strip().removesuffix('/')
    if origin.removesuffix('.git') != REPO_URL.strip().removesuffix('/').removesuffix('.git'):
        raise RuntimeError(f'{repo} was cloned from {origin}, not {REPO_URL}.')
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
os.chdir(repo)
assert sys.version_info >= (3, 11), f'Python 3.11+ is required, found {sys.version}'
GIT_COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()

def restore_attached_output(source_value, destination_value, label):
    if not source_value.strip():
        return
    source = Path(source_value).expanduser().resolve()
    destination = Path(destination_value).expanduser().resolve()
    if input_root.resolve() not in source.parents:
        raise ValueError(f'{label} source must be an attached directory under /kaggle/input: {source}')
    if not source.is_dir():
        raise FileNotFoundError(f'{label} source does not exist: {source}')
    if destination.exists() and any(destination.iterdir()):
        print(f'{label}: preserving existing writable directory {destination}')
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f'{label}: restored {source} -> {destination}')

restore_attached_output(KAGGLE_PRIOR_RUN_DIR, KAGGLE_RUN_DIR, 'Training run')
restore_attached_output(KAGGLE_PRIOR_HUMAN_DIR, HUMAN_BENCHMARK_DIR, 'Human benchmark')
print('Repository:', repo)
print('Commit:', GIT_COMMIT)

In [ ]:
%pip install -q -e '.[ml,neon,dev,notebook]'

# Make the editable src layout available without restarting the Kaggle kernel.
src_dir = str(repo / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
import importlib
for module_name in tuple(sys.modules):
    if module_name == 'sapai' or module_name.startswith('sapai.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import sapai
print('sapai:', sapai.__file__)
assert Path(sapai.__file__).resolve().is_relative_to(repo.resolve())

In [ ]:
import tensorflow as tf
from sapai.data.datasets import split_boards
from sapai.data.replay import board_is_pack_compatible
from sapai.data.serialization import read_boards
from sapai.sim.battle import BattleSimulator
from sapai.sim.catalog import Catalog

gpus = tf.config.list_physical_devices('GPU')
print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)
if REQUIRE_GPU and not gpus:
    raise RuntimeError('No GPU is visible. Select Settings -> Accelerator -> GPU and restart.')

if BOARDS_JSONL.strip():
    source_boards = Path(BOARDS_JSONL).expanduser()
else:
    attached_boards = sorted(input_root.glob('**/boards.jsonl'))
    if len(attached_boards) > 1:
        choices = '\n'.join(f'  {path}' for path in attached_boards)
        raise ValueError(f'Multiple attached boards.jsonl files found; set BOARDS_JSONL explicitly:\n{choices}')
    source_boards = (
        attached_boards[0]
        if attached_boards
        else working_root / 'sapai-data' / 'boards.jsonl'
    )
needs_board_export = not source_boards.is_file() or source_boards.stat().st_size == 0
if needs_board_export:
    resolved_source = source_boards.resolve()
    if input_root.resolve() in resolved_source.parents:
        raise ValueError(f'Attached board input is missing or empty and cannot be replaced: {source_boards}')
    try:
        from kaggle_secrets import UserSecretsClient
        database_url = UserSecretsClient().get_secret('DATABASE_URL')
    except Exception as error:
        raise RuntimeError(
            'Could not read DATABASE_URL. Add it under Kaggle Add-ons -> Secrets, '
            'or attach a dataset containing boards.jsonl.'
        ) from error
    if not database_url:
        raise RuntimeError('The DATABASE_URL Kaggle secret is empty.')
    source_boards.parent.mkdir(parents=True, exist_ok=True)
    partial_boards = source_boards.with_name(source_boards.name + '.partial')
    export_environment = os.environ.copy()
    export_environment['DATABASE_URL'] = database_url
    export_result = subprocess.run([
        sys.executable, '-m', 'sapai.cli', 'export-boards',
        '--pack', PACK, '--limit', str(BOARD_EXPORT_LIMIT),
        '--output', str(partial_boards),
    ], check=False, env=export_environment, capture_output=True, text=True)
    if export_result.returncode != 0:
        diagnostic = '\n'.join(
            part.strip() for part in (export_result.stdout, export_result.stderr) if part.strip()
        ).replace(database_url, '[REDACTED DATABASE_URL]')
        raise RuntimeError(f'Board export failed with exit code {export_result.returncode}.\n{diagnostic}')
    partial_boards.replace(source_boards)
    del database_url, export_environment
    print(f'Created board export: {source_boards}')
if not source_boards.is_file() or source_boards.stat().st_size == 0:
    raise ValueError(f'Board export did not produce data: {source_boards}')

local_boards = working_root / 'sapai-data' / 'boards.jsonl'
local_boards.parent.mkdir(parents=True, exist_ok=True)
if source_boards.resolve() != local_boards.resolve():
    shutil.copy2(source_boards, local_boards)
BOARDS_FOR_RUN = str(local_boards)

catalog = Catalog.from_json_dir(repo / 'assets' / 'data')
pack_labeled_boards = [board for board in read_boards(local_boards) if board.pack == PACK]
boards = [
    board for board in pack_labeled_boards
    if board_is_pack_compatible(board, catalog, PACK)
]
excluded_boards = len(pack_labeled_boards) - len(boards)
if excluded_boards:
    print(f'WARNING: excluded {excluded_boards:,} cross-pack boards.')
if not boards:
    raise ValueError(f'{source_boards} contains no compatible {PACK!r} boards.')
simulator = BattleSimulator(catalog)
for board in boards:
    try:
        simulator.assert_team_supported(board.team)
    except Exception as error:
        pets = [(pet.id, pet.name) for pet in board.team.slots if pet is not None]
        raise RuntimeError(
            f'Unsupported board: replay_id={board.replay_id!r}, side={board.side!r}, '
            f'turn={board.turn}, pets={pets}'
        ) from error
splits = split_boards(boards, seed=SEED)
for split_name in ('train', 'validation', 'test'):
    groups = {}
    for board in getattr(splits, split_name):
        key = (board.turn, board.pack, board.version)
        groups[key] = groups.get(key, 0) + 1
    if not any(count >= 2 for count in groups.values()):
        raise ValueError(f'{split_name} split cannot form a compatible battle pair.')
print(f'Validated {len(boards):,} {PACK} boards; writable snapshot: {local_boards}')
del boards, splits
Path(KAGGLE_RUN_DIR).parent.mkdir(parents=True, exist_ok=True)
free_gib = shutil.disk_usage(working_root).free / 2**30
print(f'Runtime disk free: {free_gib:.1f} GiB; Kaggle retains at most 20 GiB from /kaggle/working.')
if free_gib < 2:
    raise RuntimeError('Less than 2 GiB of writable runtime disk remains.')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'sapai.cli', 'model-smoke'], check=True)

## Small end-to-end smoke run

This intentionally uses a commit-specific directory and validates labeling, checkpointing, Arena rollout, and search distillation.

In [ ]:
smoke_dir = str(
    Path(KAGGLE_RUN_DIR).with_name(Path(KAGGLE_RUN_DIR).name + f'-smoke-{GIT_COMMIT[:8]}')
)
smoke = [
    sys.executable, '-m', 'sapai.cli', 'train-sequence',
    '--boards', BOARDS_FOR_RUN, '--workdir', smoke_dir, '--pack', PACK,
    '--battle-examples', '100', '--simulations-per-pair', '1',
    '--battle-epochs', '1', '--bootstrap-episodes', '2',
    '--bootstrap-epochs', '1', '--search-episodes', '1',
    '--search-epochs', '1', '--search-simulations', '4',
    '--search-candidates', '4', '--batch-size', '32', '--seed', str(SEED),
]
subprocess.run(smoke, check=True)

## Full training sequence

Set `RUN_FULL_TRAINING=True` after the smoke run succeeds. Checkpoints are written after every epoch. Kaggle sessions are limited, so save the resulting notebook version and attach its output in a later session when continuation is required. An interrupted epoch restarts from its beginning.

In [ ]:
for model_name in ('battle-model', 'policy-model'):
    checkpoint_dir = str(Path(KAGGLE_RUN_DIR) / model_name / 'checkpoints')
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        completed = int(tf.train.load_variable(
            latest_checkpoint, 'completed_epochs/.ATTRIBUTES/VARIABLE_VALUE'
        ))
        print(f'{model_name}: restoring completed epoch {completed} from {latest_checkpoint}')
    else:
        print(f'{model_name}: no checkpoint found yet')

full = [
    sys.executable, '-m', 'sapai.cli', 'train-sequence',
    '--boards', BOARDS_FOR_RUN, '--workdir', KAGGLE_RUN_DIR, '--pack', PACK,
    '--battle-examples', '100000', '--simulations-per-pair', '8',
    '--battle-epochs', '20', '--bootstrap-episodes', '1000',
    '--bootstrap-epochs', '20', '--search-episodes', '250',
    '--search-epochs', '5', '--search-simulations', '32',
    '--search-candidates', '8', '--batch-size', '128', '--seed', str(SEED),
]
if RUN_FULL_TRAINING:
    subprocess.run(full, check=True)
else:
    print('Set RUN_FULL_TRAINING=True when the smoke run succeeds.')

## Generate a portable Arena replay

The replay bundle is written beneath the active run in `/kaggle/working` and appears in the saved notebook version's Output files.

In [ ]:
active_run = KAGGLE_RUN_DIR if RUN_FULL_TRAINING else smoke_dir
visualization = str(Path(active_run) / 'arena.html')
subprocess.run([
    sys.executable, '-m', 'sapai.cli', 'visualize-arena',
    '--boards', BOARDS_FOR_RUN, '--pack', PACK, '--policy', 'model',
    '--policy-weights', str(Path(active_run) / 'policy-model'),
    '--seed', str(SEED), '--output', visualization,
], check=True)
archive = Path(active_run) / 'arena-visualization.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for name in ('arena.html', 'sapai.css', 'sapai.js'):
        bundle.write(Path(active_run) / name, name)
    for asset in (Path(active_run) / 'sapai-assets').rglob('*'):
        if asset.is_file():
            bundle.write(asset, asset.relative_to(active_run))
print(f'Visualization: {visualization}')
print(f'Portable archive: {archive}')
print('Download it from Kaggle Output, or attach this notebook output to another notebook.')

## Interactive human Arena benchmark

For an interactive editor session, set `REQUIRE_GPU=False` and `RUN_HUMAN_BENCHMARK=True`, run the setup through board validation, skip the training cells, and run the launcher below. It uses only Kaggle's bundled core `ipywidgets` models rather than a custom frontend module or Colab callbacks, while preserving the Python-authoritative session, card controls, battle review, and atomic audit files. Dragging is progressively enabled by trusted output JavaScript; **Reorder team** provides the same operation if the frontend blocks that enhancement.

After playing, save a notebook version so files beneath `HUMAN_BENCHMARK_DIR` become reusable output. On a later session, attach that output and set `KAGGLE_PRIOR_HUMAN_DIR` before running setup. Reusing compatible data resumes it; repository, simulator, or board-setting changes preserve the existing data and automatically select a deterministic suffixed directory.

In [ ]:
if RUN_HUMAN_BENCHMARK:
    from sapai.sim.shop import ShopEnvironment
    from sapai.training.human import HumanArenaSession, HumanBenchmarkConfig, sha256_file
    from sapai.training.population import load_opponent_population
    from sapai.visualization import display_human_arena_widget

    human_benchmark_dir = Path(HUMAN_BENCHMARK_DIR).expanduser().resolve()
    training_run_dir = Path(KAGGLE_RUN_DIR).expanduser().resolve()
    if (
        human_benchmark_dir == training_run_dir
        or training_run_dir in human_benchmark_dir.parents
        or human_benchmark_dir in training_run_dir.parents
    ):
        raise ValueError('HUMAN_BENCHMARK_DIR must not overlap KAGGLE_RUN_DIR.')
    human_population = load_opponent_population(BOARDS_FOR_RUN, catalog, PACK)
    human_config = HumanBenchmarkConfig(
        output_dir=human_benchmark_dir,
        participant_alias=HUMAN_PARTICIPANT_ALIAS,
        pack=PACK,
        seed=HUMAN_SEED,
        boards_sha256=sha256_file(BOARDS_FOR_RUN),
        board_count=len(human_population.boards),
        repository_commit=GIT_COMMIT,
    )
    human_session = HumanArenaSession.create_or_resume(
        ShopEnvironment(catalog),
        BattleSimulator(catalog),
        human_population,
        human_config,
        version_on_mismatch=True,
    )
    active_human_benchmark_dir = human_session.config.directory
    if active_human_benchmark_dir != human_benchmark_dir:
        print(f'Existing benchmark preserved; using compatible directory: {active_human_benchmark_dir}')
    HUMAN_WIDGET = display_human_arena_widget(human_session, repo / 'assets')
    print(f'Human benchmark artifacts: {active_human_benchmark_dir}')
else:
    print('Set RUN_HUMAN_BENCHMARK=True in an interactive session to launch human Arena.')